In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [2]:
df = pd.read_excel('../DATA JALAN PER WEEK.xlsx', usecols='B:N', engine='openpyxl')

df['Avg. Truck Per Hour'] = df.groupby('Road Segment')['Avg. Truck Per Hour'].transform(
    lambda x: x.fillna(x.median())
)
df['Avg. Truck Per Hour'] = df['Avg. Truck Per Hour'].fillna(df['Avg. Truck Per Hour'].median())
df['Speed_Deviation'] = df['Act. Speed Per Segment'] - df['Plan Speed Per Segment']

print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head()

Rows: 472, Columns: 14


,WEEK,Road Segment,Count Cycle,Avg. Truck Per Hour,Average of DISTANCE_METER,Plan Speed Per Segment,Act. Speed Per Segment,Grade,Crossfall,Lebar Jalan,Min Lebar Jalan,HRSI,Sudut Jalan,Speed_Deviation
0,26,Jalan Alphard S01,1175,29.0,0.209155,23.950000,26.13,0.045000,0.025700,25.680000,21.981,7.714286,0.0,2.180000
1,26,Jalan Alphard S02,1187,29.0,0.723199,23.950000,25.14,0.058100,0.014667,27.249667,24.735,8.875000,0.0,1.190000
2,26,Jalan Asfri S04,2,52.0,0.675945,23.950000,19.17,0.022000,0.027975,29.632750,26.025,10.000000,77.8,-4.780000
3,26,Jalan Astin S01,633,18.0,0.352394,23.950000,26.23,0.042980,0.017760,28.663400,24.000,7.000000,0.0,2.280000
4,26,Jalan Astin2 S02,113,8.0,0.404419,23.859375,24.47,0.057667,0.022033,24.858000,20.500,10.000000,0.0,0.610625


In [3]:
features = ['Grade', 'Crossfall', 'Lebar Jalan', 'Min Lebar Jalan', 
            'HRSI', 'Sudut Jalan', 'Average of DISTANCE_METER']
target = 'Act. Speed Per Segment'

X = df[features].dropna()
y = df.loc[X.index, target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test: {X_test.shape[0]} samples')
print(f'Features: {features}')

Train: 377 samples
Test: 95 samples
Features: ['Grade', 'Crossfall', 'Lebar Jalan', 'Min Lebar Jalan', 'HRSI', 'Sudut Jalan', 'Average of DISTANCE_METER']


In [4]:
try:
    from xgboost import XGBRegressor
    has_xgb = True
except ImportError:
    has_xgb = False
    print('XGBoost not installed, using GradientBoosting instead')
    from sklearn.ensemble import GradientBoostingRegressor

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
}

if has_xgb:
    models['XGBoost'] = XGBRegressor(n_estimators=100, random_state=42, max_depth=6, verbosity=0)
else:
    models['Gradient Boosting'] = GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5)

results = {}

for name, model in models.items():
    # Use scaled data for linear models, raw for tree-based
    if name in ['Linear Regression', 'Ridge', 'Lasso']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results[name] = {
        'model': model,
        'R2': r2,
        'MAE': mae,
        'RMSE': rmse,
        'CV_R2_Mean': cv_scores.mean(),
        'CV_R2_Std': cv_scores.std(),
        'y_pred': y_pred
    }
    print(f'{name}: R²={r2:.4f}, MAE={mae:.4f}, RMSE={rmse:.4f}, CV_R²={cv_scores.mean():.4f}±{cv_scores.std():.4f}')

Linear Regression: R²=0.0353, MAE=3.8274, RMSE=4.7417, CV_R²=0.0890±0.0646
Ridge: R²=0.0356, MAE=3.8277, RMSE=4.7410, CV_R²=0.0892±0.0647
Lasso: R²=0.0439, MAE=3.8312, RMSE=4.7205, CV_R²=0.0917±0.0679
Random Forest: R²=0.5945, MAE=2.3039, RMSE=3.0741, CV_R²=0.5514±0.1295
XGBoost: R²=0.6096, MAE=2.2326, RMSE=3.0164, CV_R²=0.4975±0.1254


In [ ]:
res_df = pd.DataFrame(results).T[['R2', 'MAE', 'RMSE', 'CV_R2_Mean', 'CV_R2_Std']]
res_df = res_df.sort_values('R2', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728', '#9467bd']

# R2
ax = axes[0]
bars = ax.bar(res_df.index, res_df['R2'], color=colors, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, res_df['R2']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{val:.4f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('R² Score', fontsize=11)
ax.set_title('R² Score (Higher = Better)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)

# MAE
ax = axes[1]
bars = ax.bar(res_df.index, res_df['MAE'], color=colors, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, res_df['MAE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{val:.2f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('MAE (km/h)', fontsize=11)
ax.set_title('Mean Absolute Error (Lower = Better)', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)

# RMSE
ax = axes[2]
bars = ax.bar(res_df.index, res_df['RMSE'], color=colors, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, res_df['RMSE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{val:.2f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('RMSE (km/h)', fontsize=11)
ax.set_title('Root Mean Squared Error (Lower = Better)', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Komparasi Performa Model', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n=== Ranking ===')
print(res_df.to_string())

In [ ]:
feat_imp = pd.DataFrame(index=features)

# Linear models - standardized coefficients
for name in ['Linear Regression', 'Ridge', 'Lasso']:
    coefs = np.abs(results[name]['model'].coef_)
    feat_imp[name] = coefs / coefs.sum() * 100

# Tree-based models - feature importance
for name in ['Random Forest']:
    imp = results[name]['model'].feature_importances_
    feat_imp[name] = imp / imp.sum() * 100

if has_xgb:
    imp = results['XGBoost']['model'].feature_importances_
    feat_imp['XGBoost'] = imp / imp.sum() * 100
else:
    imp = results['Gradient Boosting']['model'].feature_importances_
    feat_imp['Gradient Boosting'] = imp / imp.sum() * 100

feat_imp['Average'] = feat_imp.mean(axis=1)
feat_imp = feat_imp.sort_values('Average', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# All models heatmap
ax = axes[0]
sns.heatmap(feat_imp.drop(columns='Average'), annot=True, fmt='.1f', cmap='YlOrRd', 
            ax=ax, linewidths=0.5)
ax.set_title('Feature Importance per Model (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('')

# Average importance
ax = axes[1]
colors_imp = plt.cm.RdYlGn_r(feat_imp['Average'].values / feat_imp['Average'].max())
bars = ax.barh(feat_imp.index, feat_imp['Average'], color=colors_imp, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, feat_imp['Average']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', 
            va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Average Importance (%)', fontsize=11)
ax.set_title('Rata-rata Feature Importance\n(Semua Model)', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
best_model_name = res_df.index[0]
best_model = results[best_model_name]['model']

if best_model_name in ['Linear Regression', 'Ridge', 'Lasso']:
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Actual vs Predicted
ax = axes[0]
ax.scatter(y_test, y_pred_best, alpha=0.5, s=30, color='#1f77b4', edgecolors='white')
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
ax.plot(lims, lims, '--', color='red', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Speed (km/h)', fontsize=11)
ax.set_ylabel('Predicted Speed (km/h)', fontsize=11)
ax.set_title(f'Actual vs Predicted ({best_model_name})\nR² = {results[best_model_name]["R2"]:.4f}', 
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Residual distribution
ax = axes[1]
residuals = y_test - y_pred_best
ax.hist(residuals, bins=40, color='#1f77b4', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=2)
ax.axvline(residuals.mean(), color='orange', linestyle='--', linewidth=1.5, 
           label=f'Mean: {residuals.mean():.2f}')
ax.set_xlabel('Residual (km/h)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribusi Residual', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Cross-validation scores
ax = axes[2]
cv_data = {name: [results[name]['CV_R2_Mean'], results[name]['CV_R2_Std']] for name in results.keys()}
cv_df = pd.DataFrame(cv_data, index=['Mean', 'Std']).T
cv_df = cv_df.sort_values('Mean', ascending=True)
bars = ax.barh(cv_df.index, cv_df['Mean'], xerr=cv_df['Std'], color=colors[:len(cv_df)], 
               alpha=0.8, capsize=5, edgecolor='white')
for bar, val in zip(bars, cv_df['Mean']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}', 
            va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('CV R² Score', fontsize=11)
ax.set_title('Cross-Validation R² (5-Fold)', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
try:
    import shap
    has_shap = True
except ImportError:
    has_shap = False
    print('SHAP not installed. Install: pip install shap')

if has_shap:
    if best_model_name in ['Random Forest', 'XGBoost', 'Gradient Boosting']:
        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(X_test)
    else:
        explainer = shap.LinearExplainer(best_model, X_train_scaled)
        shap_values = explainer.shap_values(X_test_scaled)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # SHAP summary
    plt.sca(axes[0])
    shap.summary_plot(shap_values, X_test if best_model_name in ['Random Forest', 'XGBoost'] else pd.DataFrame(X_test_scaled, columns=features), 
                      feature_names=features, show=False, plot_type='bar')
    axes[0].set_title(f'SHAP Feature Importance ({best_model_name})', fontsize=12, fontweight='bold')
    
    # SHAP beeswarm
    plt.sca(axes[1])
    shap.summary_plot(shap_values, X_test if best_model_name in ['Random Forest', 'XGBoost'] else pd.DataFrame(X_test_scaled, columns=features), 
                      feature_names=features, show=False)
    axes[1].set_title(f'SHAP Beeswarm ({best_model_name})', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

In [ ]:
print('=' * 60)
print('KESIMPULAN ANALISIS')
print('=' * 60)
print(f'\nBest Model: {best_model_name}')
print(f'  R² Score: {results[best_model_name]["R2"]:.4f}')
print(f'  MAE: {results[best_model_name]["MAE"]:.4f} km/h')
print(f'  CV R²: {results[best_model_name]["CV_R2_Mean"]:.4f} ± {results[best_model_name]["CV_R2_Std"]:.4f}')
print(f'\nTop 3 Fitur Paling Penting:')
top3 = feat_imp['Average'].tail(3)
for i, (feat, imp) in enumerate(enumerate(top3), 1):
    print(f'  {i}. {top3.index[3-i]}: {top3.values[3-i]:.1f}%')
print(f'\nInterpretasi:')
print(f'  - Model dapat menjelaskan {results[best_model_name]["R2"]*100:.1f}% variansi Actual Speed')
print(f'  - Error rata-rata: {results[best_model_name]["MAE"]:.2f} km/h')
print(f'  - Fitur paling berpengaruh: {top3.index[-1]}')